# 6.1 Exception Handling

**Prerequisites:** 05 OOPs (exceptions are classes)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Errors vs exceptions; syntax, logical and run-time errors
- **The exception hierarchy**, and why `except Exception` is the safe ceiling
- `try` / `except` / `else` / `finally` — what each block is actually for
- **Reading a traceback** bottom-up
- **The bare `except:` anti-pattern**, and `contextlib.suppress`
- Re-raising with a bare `raise`, and `logging.exception()`
- 🔴 **Why `assert` must never validate input** — `-O` removes it
- Raising exceptions, and a first look at custom exception classes

---

### Error: 
- Error is an illegal operation performed by the programmer/user which results in abnormal working of the program.
- **Error Types:** Errors are generally classified as,
    - Syntax Error or Parsing Error or Compilation Error
    - Logical Error 
    - Run-time Error
<img src='./Image/6.1 Image a.jpg' width=90% height=90%/>
<br/><br/>
- **NOTE: Handling of Errors,**
    - Syntax Error and Logical Error can only be handled by programmer.
    - But Run-time Error can be handled by either programmer or PVM.

In [ ]:
# Example of syntax error
print('Start of program')
print('Hello) # Syntax Error: Missed single quote at the end of string
print('End of program')

In [ ]:
# Example of logical error
print('Start of program')
n = int(input("Enter a number: "))
if n%2!=0: # Logical Error
    print("Even")
else:
    print("Odd")
print('End of program')

In [ ]:
# Example of Run-time error:
print('Start of program')
n1 = int(input("Enter a number: "))
n2 = int(input("Enter a number: "))
div = n1/n2 # Exceptionally error will occur only when n2=0
print(div)
print('End of program')

### Types of program termination:
- **Normal Termination:** Program end without any problem.
- **Abnormal Termination:** Program end with problem i.e, the problem disrupts the flow of the program. These kind of problem are known as Exceptions.


## Exceptions
- Errors detected during execution are called exceptions.
    - Exception means something that is not expected.
- An exception can be defined as raising of abnormal condition which disrupts the normal flow of the program.
    - Whenever an exception occurs, the program halts the execution, and thus the further code is not executed.
- Python provides us with the way to handle the Exception so that the other part of the code can be executed without any disruption. 
    - However, if we do not handle the Exception, PVM handles the Exception but then it doesn't execute all the code that exists after the statement which raised Exception.
- Exception handling means creating events that can be used to modify the flow of control when an error occurs in the program. 
    - Event get triggered automatically on finding error of specific type.

### Built-in Python Exceptions:
- 1) **Exception:** This is the base class for all kind of the exceptions. 
    - All kind of exceptions should be derived from this class
<br/><br/>
- 2) **ArithmeticError:** This is the base class for the exception raised for any arithmetic errors.
<br/><br/>
- 3) **ZeroDivisionError:** This exception raise when the second argument of a division or modulo operation is zero
<br/><br/>
- 4) **FloatingPointError:** This exception raise when a floating point operation fails.
<br/><br/>
- 5) **NameError:** This exception raise when a name is not found. It may be local or global.
<br/><br/>
- 6) **IndexError** : This exception raise when the index out of range.
<br/><br/>
- 7) **ValueError** : This exception raise due to invalid literal.
<br/><br/>
- 8) **AttributeError** : This exception raise when using a function which doesn't exist.
<br/><br/>
- 9) **TypeError** : This exception raise when operation performed on wrong type.
<br/><br/>
- 10) **EOFError:** This exception raise when the end of the file is reached, and yet operations are being performed.
<br/><br/>
- 11) **IndentationError:** This exception raise when incorrect indentation is given.
<br/><br/>
- 12) **AssertionError:** This exception raise when an assert statement fails.
<br/><br/>
- 13) **KeyError:** This exception raise when a mapping (dictionary) key is not found in the set of existing keys.
<br/><br/>
- 14) **IOError:** This exception raise when Input Output operation fails. 
    - Ex. open function when try to open a file that doesn't exist.
<br/><br/>    
- 15) **KeyboardInterrupt:** This exception raise when the user hits the interrupt key (normally Control-C or Delete). During execution, a check for interrupts is made regularly.
<br/><br/>
- In Python, all Exceptions must be instances of a class that is derives from BaseException.
<img src='./Image/6.1 Image b.jpg' width=80% height=80%/>
- The class hierarchy for built-in exceptions:
<https://docs.python.org/3/library/exceptions.html#bltin-exceptions>

### The exception hierarchy

Exceptions are classes, so they form an inheritance tree — and `except` matches
**subclasses** as well as the exact type. That is what lets one handler cover a family of
related failures.

```
BaseException
 ├── SystemExit             sys.exit()
 ├── KeyboardInterrupt      Ctrl-C
 ├── GeneratorExit          generator .close()
 └── Exception              <- everything you normally catch
      ├── ArithmeticError -> ZeroDivisionError, OverflowError
      ├── LookupError     -> IndexError, KeyError
      ├── OSError         -> FileNotFoundError, PermissionError, ConnectionError
      ├── ValueError, TypeError, AttributeError, NameError, ...
```

Two consequences worth remembering:

1. **`except LookupError` catches both `IndexError` and `KeyError`** — because both inherit
   from it. Catch the level of the tree that matches what you can actually handle.
2. **`SystemExit`, `KeyboardInterrupt` and `GeneratorExit` are deliberately *not* under
   `Exception`.** They are control-flow signals, not errors. That is precisely why
   `except Exception:` is safe and a bare `except:` is not.

In [ ]:
# Every exception you can catch descends from BaseException.
print("BaseException")
print("├── SystemExit          <- sys.exit()")
print("├── KeyboardInterrupt   <- Ctrl-C")
print("├── GeneratorExit       <- generator.close()")
print("└── Exception           <- EVERYTHING you normally catch")
print("    ├── ArithmeticError")
print("    │   └── ZeroDivisionError")
print("    ├── LookupError")
print("    │   ├── IndexError")
print("    │   └── KeyError")
print("    ├── OSError")
print("    │   ├── FileNotFoundError")
print("    │   ├── PermissionError")
print("    │   └── ConnectionError")
print("    ├── ValueError")
print("    ├── TypeError")
print("    ├── AttributeError")
print("    └── ... and many more")

# The hierarchy is why you can catch a whole family at once
for exc_type in (IndexError, KeyError):
    print(f"\n{exc_type.__name__} is a LookupError:", issubclass(exc_type, LookupError))

# One handler for both:
for container, key in [([1, 2, 3], 10), ({"a": 1}, "b")]:
    try:
        container[key]
    except LookupError as exc:
        print(f"  caught {type(exc).__name__} via LookupError")

# See any exception's ancestry
print("\nFileNotFoundError MRO:")
print("  " + " -> ".join(cls.__name__ for cls in FileNotFoundError.__mro__[:-1]))

# ⚠️ The three that are NOT under Exception
print("\nNot subclasses of Exception (deliberately):")
for exc_type in (SystemExit, KeyboardInterrupt, GeneratorExit):
    print(f"  {exc_type.__name__:<18} issubclass(..., Exception) = "
          f"{issubclass(exc_type, Exception)}")

In [ ]:
# import builtins
# help(builtins)

In [ ]:
# print(1/0) #ZeroDivisionError

In [ ]:
# print('a'/5) #TypeError

In [ ]:
# print(xyz) #NameError

In [ ]:
# d={}
# print(d['a']) #KeyError

In [ ]:
# l = [1,2,3]
# print(l[10]) # IndexError

In [ ]:
# a=20
# if a>10:
#     print(a)
#  print(a+2) #IndentationError

In [ ]:
# l = [1,2,3]
# l.pick()  #AttributeError

In [ ]:
# file= open('file.txt') #FileNotFoundError
# print(file.read())

---

### Reading a traceback

A traceback is not noise — it is a precise report, and learning to read one is the single
most useful debugging skill there is.

**Read it bottom-up:**

1. **The last line** names the exception type and message — *what* went wrong.
2. **The frame just above it** is the line that actually raised — *where*.
3. **Everything above that** is the call chain that got you there — *how*.

The most common mistake is reading a traceback top-down and fixating on the first file
name, which is usually just `<module>` or a framework entry point.

In [ ]:
import traceback


def load_settings(path: str) -> dict:
    return parse_file(path)


def parse_file(path: str) -> dict:
    raw = read_file(path)
    return {"timeout": int(raw)}


def read_file(path: str) -> str:
    return "not-a-number"          # the actual source of the problem


try:
    load_settings("app.conf")
except ValueError:
    print(traceback.format_exc())

print("""
Read it bottom-up:

  ValueError: invalid literal for int() ...   <- WHAT went wrong (read this first)
  File "...", line N, in parse_file           <- WHERE it went wrong
  File "...", line N, in load_settings        <- who called that
  File "...", line N, in <module>             <- where it all started

The last line names the error. The frame directly above it is the line
that raised. Everything above that is how you got there.
""")

# ---- Inspecting a traceback programmatically ----
try:
    load_settings("app.conf")
except ValueError as exc:
    tb = exc.__traceback__
    frames = traceback.extract_tb(tb)
    print("call chain, outermost first:")
    for frame in frames:
        print(f"  {frame.name:<15} line {frame.lineno}")
    print(f"\nthe line that raised: {frames[-1].line}")

### Exception Handling
- While writing the code, some statements might be suspicious for raising an error.
- The Exceptions in python are processed using five statements. They are:
    - 1. **try**
    - 2. **except**
    - 3. **finally**
    - 4. **else**
    - 5. **raise**

### Python try except else finally: 
- If the python program contains suspicious code that may throw an Exception, we must place that code in the try block. 
- The **try block** must be followed with the **except block** which contains a block of code that will be executed if there is some Exception raised in the try block.
- We can also use the **else block** with the try-except statement in which, we can place the code which will be executed in the scenario if no exception occurs in the try block.
- We can use the **finally block** with the try block in which, we can pace the important code which must be executed before the try statement throws an exception.
<br/><br/>
- **Syntax:**

```python
try :
    block of code  # doubtful code

except Exception_class1:
    block of code  # this code executes if exception occurs in the try block
  
except Exception_class2: 
    block of code  # this code executes if first except block was unable to handled the exception
    
else:  
    block of code # this code executes only if no exception occurs in the try block

finally:
    block of code  # this will always be executed
    
other code
```
  

### Points to remember:
- 1) The try block lets you test a block of code for errors.
- 2) The except block lets you handle the error. We can declare multiple except statement.
- 3) The else block run if no exception occurs.
- 4) The finally block lets you execute code, regardless of the result of the try and except blocks.

In [ ]:
# Handling ZeroDivision Error:
print('Start of program')
n1 = int(input("Enter a number: "))
n2 = int(input("Enter a number: "))
try:
    # Write the suspicious block of code
    div = n1/n2 # error will occur only when n2=0
    print(div)
except ZeroDivisionError as e: # catching exception inside object e of ZeroDivisionError class
    print(e)
print('End of program')

In [ ]:
# Handling ZeroDivision Error But instead of 0 user inputted non-numeric value:
print('Start of program')
# Suppose user inputted non-numeric value,
n1 = int(input("Enter a number: "))
n2 = int(input("Enter a number: "))
try:
    div = n1/n2
    print(div)
except ZeroDivisionError as e: # catching exception inside object e of ZeroDivisionError class
    print(e)
print('End of program')

In [ ]:
# Handling Multiple Exceptions using Multiple Except statement:
print('Start of program')
try:
    n1 = int(input("Enter a number: "))
    n2 = int(input("Enter a number: "))
    div = n1/n2
    print(div)
    
except ZeroDivisionError as e1: # catching exception inside object e1 of ZeroDivisionError class
    print(e1)
except ValueError as e2: # catching exception inside object e2 of ValueError class
    print(e2)
print('End of program')
# Raised Exception class will be searched in defined order inside except block

In [ ]:
# Handling Multiple Exceptions using one except statement
print('Start of program')
try:
    n1 = int(input("Enter a number: "))
    n2 = int(input("Enter a number: "))
    div = n1/n2
    print(div)
except (ZeroDivisionError,ValueError) as e: # catching exception inside object e by declare multiple exception classes
    print(e)
print('End of program')

In [ ]:
# Handling exception Gracefully:
print('Start of program')
try:
    n1 = int(input("Enter a number: "))
    n2 = int(input("Enter a number: "))
    div = n1/n2
    print(div)
except (ZeroDivisionError,ValueError):
    print('Please provide input carefully')
print('End of program')

- Instead of declaring multiple Exception child classes we can directly declare base ```Exception``` class, which is capable of handling any kind of exception.
    - May increase the time of execution as error will be seached in hierarchy of Exception class defined.
- But it is never recommended to use it directly.

In [ ]:
# Handling Multiple Exceptions using Base Exception class
print('Start of program')
try:
    n1 = int(input("Enter a number: "))
    n2 = int(input("Enter a number: "))
    div = n1/n2 # error will occur only when n2=0
    print(div)
except Exception as e: # catching exception inside object e by declare multiple exception classes
    print(e)
print('End of program')

In [ ]:
# The finally block, if specified, will be executed regardless if the try block raises an error or not.
try:
    a = int(input("Enter your 1st no. "))
    b = int(input("Enter your 2nd no. "))
    div = a/b
    print(div)
except (ZeroDivisionError, ValueError):
    print ("Divided by zero error or  datatype error")
else:
    print('Else bock')
finally:
    print('End of program')

---

### ⚠️ The bare `except:` anti-pattern

You will see this everywhere. It is almost always wrong:

```python
try:
    do_something()
except:            # ⚠️ catches absolutely everything
    pass
```

A bare `except:` catches **`BaseException`**, which includes:

- **`KeyboardInterrupt`** — so Ctrl-C stops working
- **`SystemExit`** — so `sys.exit()` stops working
- **`MemoryError`**, `GeneratorExit`, and anything else

It also hides your own bugs: a `NameError` from a typo is caught and discarded exactly like
the error you were expecting.

| Instead of | Write |
|---|---|
| `except:` | `except Exception:` — ordinary errors only |
| `except Exception: pass` | `contextlib.suppress(SpecificError)` — explicit |
| `except Exception:` around everything | The narrowest exception you can actually handle |

The rule: **catch what you can handle.** If you cannot do anything useful with the error,
let it propagate — a crash with a traceback is far better than silent wrong behaviour.

In [ ]:
# ---- ⚠️ Bare except: catches things you never meant to catch ----
import sys

class FakeInterrupt(BaseException):
    """Stands in for KeyboardInterrupt, which we cannot press here."""


def bare_except():
    try:
        raise FakeInterrupt("user pressed Ctrl-C")
    except:                       # ⚠️ catches EVERYTHING, including BaseException
        return "swallowed - the user could not quit"


def correct_except():
    try:
        raise FakeInterrupt("user pressed Ctrl-C")
    except Exception:             # ✅ ordinary errors only
        return "swallowed"


print("bare except:", bare_except())

try:
    correct_except()
except FakeInterrupt as exc:
    print("except Exception: interrupt propagated correctly ->", exc)


# ---- The hierarchy in action ----
print("\nKeyboardInterrupt is an Exception?", issubclass(KeyboardInterrupt, Exception))
print("KeyboardInterrupt is a BaseException?", issubclass(KeyboardInterrupt, BaseException))
print("ValueError is an Exception?         ", issubclass(ValueError, Exception))


# ---- `except Exception: pass` hides real bugs ----
def silently_broken(data):
    try:
        return data["totl"]           # typo - should be "total"
    except Exception:
        pass                          # the typo is now invisible forever

print("\nsilently_broken:", silently_broken({"total": 100}), " <- None, and nobody knows why")


# ---- If you genuinely mean "ignore this", say so ----
from contextlib import suppress

value = None
with suppress(KeyError):              # explicit, narrow, and self-documenting
    value = {"total": 100}["missing"]
print("suppress(KeyError) ->", value)

### `else` and `finally`, precisely

Four blocks, and each has one job:

| Block | Runs when | Use it for |
|---|---|---|
| `try` | Always | **Only** the risky operation — keep it small |
| `except` | An exception matched | Handling that specific failure |
| `else` | The `try` block raised **nothing** | The rest of the happy path |
| `finally` | **Always**, even on `return` or re-raise | Cleanup |

### Why `else` exists

Without it, the happy-path code sits inside `try` — where it is also covered by the
handlers. If that code happens to raise the same exception type, your handler catches it
and reports the wrong cause.

```python
try:
    value = int(raw)          # the risky bit
    result = lookup(value)    # ⚠️ if this raises ValueError too, the handler lies
except ValueError:
    ...

try:
    value = int(raw)          # ✅ only the risky bit
except ValueError:
    ...
else:
    result = lookup(value)    # a ValueError here propagates honestly
```

### 🔴 Never `return`, `break` or `continue` from `finally`

Because `finally` always runs, a `return` inside it **discards any exception that was in
flight** and overrides any earlier return value:

```python
def broken():
    try:
        raise ValueError("something went wrong")
    finally:
        return "finally wins"     # the ValueError vanishes without trace
```

This was a silent, notorious bug source for decades. Python has been progressively closing
it:

> ### Version note — PEP 765
> | Version | `return` / `break` / `continue` in `finally` |
> |---|---|
> | ≤ 3.11 | Allowed, silently swallows exceptions |
> | **3.12 – 3.13** | **`SyntaxWarning`** |
> | **3.14+** | **`SyntaxError` — the file will not even compile** |

So on a modern interpreter this mistake is caught for you. On anything older it is not, and
you will still meet the pattern in existing code. The cell below detects which behaviour
your interpreter has.

In [ ]:
import sys, warnings

print("running on Python", ".".join(map(str, sys.version_info[:3])))

BAD = """
def broken():
    try:
        raise ValueError("something went wrong")
    finally:
        return "finally wins"
"""

# Compile it rather than defining it, so this cell works on every version.
with warnings.catch_warnings():
    warnings.simplefilter("error")          # turn SyntaxWarning into an exception
    try:
        compile(BAD, "<demo>", "exec")
        print("  -> compiled silently (Python <= 3.11 behaviour)")
        verdict = "allowed"
    except SyntaxWarning as warn:
        print("  -> SyntaxWarning:", warn)
        verdict = "warned"
    except SyntaxError as exc:
        print("  -> SyntaxError:", exc.msg)
        verdict = "rejected"

print(f"\nverdict: return-in-finally is {verdict} here")

# break and continue are treated identically
for keyword in ("break", "continue"):
    src = f"""
def g():
    for i in range(3):
        try:
            pass
        finally:
            {keyword}
"""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            compile(src, "<demo>", "exec")
        print(f"  {keyword:<8} in finally: allowed")
    except (SyntaxError, SyntaxWarning) as exc:
        msg = getattr(exc, "msg", str(exc))
        print(f"  {keyword:<8} in finally: {type(exc).__name__} - {msg}")


# ---- What to write instead ----
def correct(raw: str) -> str:
    result = "default"
    try:
        result = str(int(raw))
    except ValueError:
        result = "invalid"
    finally:
        print("  finally: cleanup only, no control flow")
    return result                    # return AFTER the try statement, not inside finally


print("\ncorrect('42') ->", correct("42"))
print("correct('ab') ->", correct("ab"))

### Python `assert`

An `assert` statement takes a condition. If the condition is falsy, it raises
`AssertionError` with an optional message.

```
assert condition, "message shown when it fails"
```

It is shorthand for:

```python
if __debug__ and not condition:
    raise AssertionError("message shown when it fails")
```

### 🔴 The critical caveat: assertions can be switched off

**When Python runs with the `-O` (optimise) flag, every `assert` statement is removed from
the compiled bytecode.** It does not evaluate to `True` — it ceases to exist.

```bash
python  script.py      # assertions active
python -O script.py    # assertions GONE
```

That makes `assert` **unsafe for anything that must always happen**. If you validate user
input with an assertion, then deploy with `-O`, the validation silently vanishes and the bad
value flows straight through.

| Use `assert` for | Use `raise` for |
|---|---|
| Internal invariants — "this should be impossible" | Validating user or caller input |
| Sanity checks during development | Enforcing a documented precondition |
| Tests (`pytest` is built on `assert`) | Anything a library caller could trigger |
| Documenting an assumption to a reader | Anything security- or money-related |

**The rule:** if the check protects against a *bug in your own code*, `assert` is fine. If it
protects against *the outside world*, raise a real exception.

> The example below is exactly the wrong use — validating an argument. It is kept because
> you will see this pattern in the wild, followed immediately by the correct version.

In [ ]:
def get_age1(age):
    print("Your age is:", age)
    
get_age1(-1)

In [ ]:
# Example of assert to Raise error:
def get_age2(age):
    assert age>0, "Age must be greater than 0"
    print("Your age is:", age)

try:
    a= int(input('Enter your age: '))
    get_age2(a)
except AssertionError as e:
    print(e)

In [ ]:
import subprocess, sys, textwrap, pathlib

# ---- The WRONG way: validation by assertion ----
def set_discount_unsafe(percent: float) -> float:
    assert 0 <= percent <= 100, "discount must be between 0 and 100"
    return percent


# ---- The RIGHT way: an explicit raise ----
def set_discount(percent: float) -> float:
    if not 0 <= percent <= 100:
        raise ValueError(f"discount must be between 0 and 100, got {percent}")
    return percent


for func, label in [(set_discount_unsafe, "assert"), (set_discount, "raise ")]:
    try:
        func(150)
    except (AssertionError, ValueError) as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")

# ---- Now prove that -O removes the assertion ----
script = textwrap.dedent("""
    def set_discount_unsafe(percent):
        assert 0 <= percent <= 100, "discount must be between 0 and 100"
        return percent

    print("  accepted:", set_discount_unsafe(150))
""")

demo = pathlib.Path("assert_demo.py")
demo.write_text(script, encoding="utf-8")

for flags, label in [([], "python    "), (["-O"], "python -O ")]:
    result = subprocess.run(
        [sys.executable, *flags, str(demo)],
        capture_output=True, text=True,
    )
    outcome = result.stdout.strip() or result.stderr.strip().splitlines()[-1]
    print(f"\n{label}: {outcome}")

demo.unlink()

print("""
Under -O the assertion is gone and the invalid discount of 150 is accepted.
An explicit `raise` is unaffected by -O, which is why validation must use one.
""")

### Raising Exception:
- In python we can raise an exception explicitely using 'raise' keyword.
- The raise statement allows the programmer to force a specific exception to occur. 
- The sole argument in raise indicates the exception to be raised. 
- This must be either an exception instance or an exception class (a class that derives from Exception).
- **Syntax:**

```python    
raise Exception_class,<value>
``` 

### Points to remember:
- 1) To raise an exception, raise statement is used. The exception class name follows it.
- 2) An exception can be provided with a value that can be given in the parenthesis.
- 3) To access the value "as" keyword is used. "e" is used as a reference variable which stores the value of the exception.

In [ ]:
def check_age(age):
    if age<0:
        raise ValueError('Age input is less than 0')
    elif isinstance(age, float):
        raise ValueError('Age must be in integer')
        
try:
    ip_age = eval(input("Enter age: "))
    check_age(ip_age)
    print(ip_age)
except ValueError as e:
    print(e)

In [ ]:
# Custom Exception class:
class Check_price(Exception):
    def __init__(self,msg):
        super().__init__(msg)

def calculate_discount(price,disc):
    if price<0 or disc<0:
        raise Check_price('Price or Disc can\'t be less than zero')
    else:
        discount = price-((price*disc)/100)
        return discount
try:    
    ip_price = float(input("Enter price: "))
    ip_disc =  float(input("Enter discount: "))
    print(calculate_discount(ip_price,ip_disc))
except Check_price as e:
    print(e)

In [ ]:
class Check_price(Exception):
    def __init__(self,*args):
        if args:
            self.message= args[0]
        else:
            self.message= None
            
    def __str__(self):
        "Calling str"
        if self.message:
            return f'CustomError: {self.message}'
        else:
            return f'CustomError: An error occured'

def calculate_discount(price,disc):
    if price<0 or disc<0:
        raise Check_price('Price or Disc can\'t be less than zero')
    else:
        discount = price-((price*disc)/100)
        return discount
try:    
    ip_price = float(input("Enter price: "))
    ip_disc =  float(input("Enter discount: "))
    print(calculate_discount(ip_price,ip_disc))
except Check_price as e:
    print(e)

---

### Re-raising, and logging an exception properly

Two habits that separate debuggable code from the other kind.

**1. A bare `raise` re-raises the current exception** with its original traceback intact.
`raise exc` also works but resets the traceback to the current line, losing where it
actually came from.

**2. `logging.exception()` records the traceback; `print(exc)` throws it away.**
Inside an `except` block, `log.exception("what I was doing")` writes your message *and* the
full traceback at ERROR level. This is the single highest-value habit in this notebook.

> Exceptions are ordinary objects. `exc.args` holds the constructor arguments, and you can
> attach your own attributes — which is the basis of custom exceptions in **6.2**.

⚠️ One surprise: the name bound by `except ... as exc` is **deleted** at the end of the
block, to break a reference cycle. If you need the exception afterwards, assign it to
another name.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
log = logging.getLogger("orders")


def fetch_order(order_id: int) -> dict:
    orders = {1: {"id": 1, "total": 250.0}}
    return orders[order_id]          # raises KeyError for anything else


# ---- print() loses the traceback; logging.exception() keeps it ----
try:
    fetch_order(99)
except KeyError as exc:
    print("with print()     :", exc, " <- just the key, no context at all")

try:
    fetch_order(99)
except KeyError:
    log.exception("could not load order")     # logs message AND full traceback

print("\n" + "-" * 50)


# ---- A bare `raise` re-raises the CURRENT exception, traceback intact ----
def load_order(order_id: int) -> dict:
    try:
        return fetch_order(order_id)
    except KeyError:
        log.warning("order %s not found, re-raising", order_id)
        raise                       # bare raise - preserves the original traceback


try:
    load_order(99)
except KeyError as exc:
    print("re-raised, still a KeyError:", repr(exc))


# ---- Exceptions are objects: they can carry data ----
try:
    int("not a number")
except ValueError as exc:
    print("\ntype    :", type(exc).__name__)
    print("args    :", exc.args)
    print("str()   :", str(exc))
    print("repr()  :", repr(exc))

# ⚠️ The name bound by `as` is deleted when the except block ends
try:
    1 / 0
except ZeroDivisionError as exc:
    saved = exc                     # keep a reference if you need it later
print("\nafter the block, `exc` exists?", "exc" in dir())
print("but `saved` does:", repr(saved))

---

## Common Mistakes & Pitfalls

1. 🔴 **Using `assert` to validate input.** Assertions are removed entirely under `python -O`. Use `if ...: raise ValueError(...)`.
2. **Bare `except:`.** It catches `KeyboardInterrupt` and `SystemExit` too, so Ctrl-C stops working. Write `except Exception:` at minimum.
3. **Catching an exception and doing nothing** (`except Exception: pass`). The failure is now invisible. If you really mean it, use `contextlib.suppress` so the intent is explicit.
4. **Catching too broadly, too early.** `except Exception` around a whole function hides typos (`NameError`, `AttributeError`) as if they were expected failures.
5. **Losing the original error** by raising a new one without `from` (see **6.2**).
6. **`return` inside `finally`.** It discards any in-flight exception *and* any earlier return value — silently.
7. **Putting cleanup after the `try` block instead of in `finally`.** If the block raises, the cleanup never runs.
8. **Using exceptions for ordinary control flow** where a simple `if` would do. (`dict.get()` beats `try: d[k] except KeyError`.)
9. **Printing an exception instead of logging it.** `print(exc)` loses the traceback; `logging.exception()` keeps it.

## Best Practices

- Catch the **most specific** exception that you can actually handle.
- Keep the `try` block **as small as possible** — ideally the single risky call.
- Use `else` for the code that should run only when nothing was raised.
- Use `finally` (or better, a context manager — **6.3**) for cleanup.
- Re-raise with a bare `raise` to preserve the original traceback.
- Use `logging.exception()` inside an `except` block; it records the traceback automatically.
- Prefer **EAFP** (`try`/`except`) over pre-checking, but use `dict.get()` and similar when the language already gives you a non-raising option.
- Reserve `assert` for internal invariants and tests — never for validating external input.

## Practice Exercises

Try these before moving on.

1. Write a function that reads an integer from a string, handling `ValueError` and returning a default. Then rewrite it so the caller decides what to do.
2. Show that a bare `except:` swallows `KeyboardInterrupt`, then fix it.
3. Write a `try/except/else/finally` where all four blocks print, and predict the order for both the success and failure paths.
4. Write a function with `return` in both `try` and `finally`, and explain the result.
5. Run a script with `assert x > 0` under `python -O` and confirm the check disappears.
6. Read a traceback from a three-level call and identify which frame actually raised.
7. Rewrite a nested `try/except` chain using `else` so the happy path is unindented.
8. Use `logging.exception()` in a handler and compare the output with `print(exc)`.